[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/MFM/blob/main/Quantlets/Ch_08/MFM_ch8_sem_partA/MFM_ch8_sem_partA.ipynb)

# MFM_ch8_sem_partA

Seminar 8, Part A: local and exact power of the Kupiec test, days needed for a given power, the Christoffersen Markov-chain likelihood, the DQ statistic with a constant as the score form of Kupiec, traffic-light zones, numerical check of the consistency of the pinball and FZ0 losses, a counterexample to the elicitability of ES, Acerbi-Szekely Z1 and Z2, moments of the Du-Escanciano cumulative violation, and a conformal quantile with one adaptive conformal step.

*Modelling Financial Markets (MFM), Chapter 8: Backtesting and Evaluating Risk Forecasts. Author: Daniel Traian Pele.*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/MFM/Chapter_8'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
import os
import sys
import subprocess
try:
    import arch  # GARCH estimation
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'arch'], check=True)
import pickle
import tempfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.colors import LinearSegmentedColormap
from scipy import stats, optimize
from numpy.lib.stride_tricks import sliding_window_view
import warnings
warnings.filterwarnings('ignore')

## Data and backtesting tools

Market prices from the course repository (`data/market`); EUR/RON: BNR reference rate. Rolling one-day-ahead VaR/ES forecasts (1000-day window) and all backtests.

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/MFM/main/data/market/'
# copia locala a folderului data/market, daca notebook-ul ruleaza din repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')

# name -> (symbol, start date); indices/crypto: closing price
SOURCES = {
    'sp500':  ('GSPC.INDX', '1990-01-01'),
    'bet':    ('BET', '1997-09-19'),
    'btc':    ('BTC-USD.CC', '2014-09-17'),
    'eurron': ('REF:EUR', '2005-07-01'),        # BNR reference rate, since the introduction of the new leu
}
LABELS = {'sp500': 'S&P 500', 'bet': 'BET', 'btc': 'Bitcoin', 'eurron': 'EUR/RON'}
ASSETS = list(SOURCES)

_CACHE = {}


def read_market(symbol):
    """Read data/market/<SYMBOL>.csv locally or from the GitHub repository."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname)
    src = path if os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def read_reference_rate(currency='EUR', start='2005-07-01', end='2026-09-18'):
    """Official RON reference rate published by the BNR (yearly XML archives)."""
    key = (currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    import re
    import urllib.request
    rows = []
    for y in range(int(start[:4]), int(end[:4]) + 1):
        url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
        xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}">([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(1))))
    s = pd.DataFrame(rows, columns=['date', 'close']).drop_duplicates('date').set_index('date')['close']
    s.index = pd.to_datetime(s.index)
    _CACHE[key] = s.sort_index().loc[start:end]
    return _CACHE[key]


def load_price(name):
    """Price series (close) of the asset."""
    symbol, start = SOURCES[name]
    if symbol.startswith('REF:'):
        return read_reference_rate(symbol.split(':')[1], start=start)
    p = read_market(symbol)['close'].loc[start:]
    return p[p > 0].dropna()


def load_returns(name):
    """Daily log returns on the series' own calendar."""
    return np.log(load_price(name)).diff().dropna()


# =============================================================================
# VaR / ES FORECASTS ON A ROLLING WINDOW (one day ahead)
# =============================================================================
W = 1000                       # estimation window (observations)
REFIT = 20                     # GARCH / t / EVT parameters are re-estimated every 20 days
LEVELS = (0.01, 0.025, 0.05)   # tail probabilities: VaR 1%, VaR 2.5%, VaR 5%
A_ES = 0.025                   # ES 2.5% (FRTB)
K_TAIL = int(np.ceil(A_ES * W))   # number of tail order statistics (Z2 simulation)
MODELS = ['HS', 'Normal', 'Student-t', 'GARCH-t', 'FHS', 'GARCH-EVT']
EVAL_FROM = '2007-01-01'       # first possible evaluation day
EVT_Q = 0.90                   # POT threshold: 90% quantile of the standardised losses


def t_std_q(nu, a):
    """Quantile of order 1-a of the standardised Student-t distribution (unit variance)."""
    return stats.t.ppf(1 - a, nu) * np.sqrt((nu - 2) / nu)


def t_std_es(nu, a):
    """ES at level 1-a of the standardised Student-t distribution (unit variance)."""
    q = stats.t.ppf(1 - a, nu)
    return stats.t.pdf(q, nu) / a * (nu + q ** 2) / (nu - 1) * np.sqrt((nu - 2) / nu)


def garch_fit(x, prev=None):
    """GARCH(1,1) with Student-t innovations by maximum likelihood; parameters in decimal units.
    The series is rescaled to unit variance for numerical stability; if the optimisation fails,
    the previous estimates are kept."""
    from arch import arch_model
    c = 1 / x.std()
    res = arch_model(c * x, mean='Constant', vol='GARCH', p=1, q=1, dist='t',
                     rescale=False).fit(disp='off', show_warning=False)
    mu, om, a, b, nu = res.params.values
    ok = res.convergence_flag == 0 and abs(mu) < 1 and a + b < 1 and om > 0
    if not ok and prev is not None:
        return prev
    return mu / c, om / c ** 2, a, b, nu


def garch_filter(x, mu, om, a, b, s2_0):
    """Conditional variance: s2[t] uses only x[:t]; the last element is the forecast."""
    e2 = (x - mu) ** 2
    s2 = np.empty(len(x) + 1)
    s2[0] = s2_0
    for t in range(len(x)):
        s2[t + 1] = om + a * e2[t] + b * s2[t]
    return s2


def gpd_tail(y, a_list, q=EVT_Q):
    """EVT (POT): GPD on the excesses of standardised losses over the q quantile; VaR/ES at levels 1-a."""
    u = np.quantile(y, q)
    exc = y[y > u] - u
    xi, _, beta = stats.genpareto.fit(exc, floc=0)
    pu = len(exc) / len(y)
    var = {a: u + beta / xi * ((a / pu) ** (-xi) - 1) for a in a_list}
    es = {a: (var[a] + beta - xi * u) / (1 - xi) for a in a_list}
    return dict(u=u, xi=xi, beta=beta, pu=pu, var=var, es=es)


def gpd_cdf(y, g, emp):
    """EVT distribution function: empirical below the threshold, GPD above it."""
    if y <= g['u']:
        return np.mean(emp <= y)
    base = 1 + g['xi'] * (y - g['u']) / g['beta']
    return 1.0 if base <= 0 else 1 - g['pu'] * base ** (-1 / g['xi'])


def rolling_forecasts(r, eval_from=EVAL_FROM, w=W, refit=REFIT):
    """One-day-ahead VaR (LEVELS) and ES (A_ES) forecasts for the loss L = -r, for all models.

    Result: dict model -> DataFrame (index = forecast day) with columns
    VaR1, VaR2.5, VaR5, ES2.5, ES1, pit (F(L_t)), scale, mu, plus information for tail simulation."""
    r = r.dropna()
    L = -r.values
    idx = r.index
    start = max(w, int(np.searchsorted(idx, pd.Timestamp(eval_from))))
    T = len(r) - start
    days = idx[start:]
    out = {m: {k: np.full(T, np.nan) for k in ['VaR1', 'VaR2.5', 'VaR5', 'ES2.5', 'ES1', 'pit', 'scale', 'mu', 'nu',
                                               'u', 'xi', 'beta', 'pu']} for m in MODELS}
    tails = {m: np.full((T, K_TAIL), np.nan) for m in ('HS', 'FHS')}
    lev = {0.01: 'VaR1', 0.025: 'VaR2.5', 0.05: 'VaR5'}
    Lcur = L[start:]

    # --- HS and Normal: vectorised over windows
    win = sliding_window_view(L[:-1], w)[start - w:]          # window for day t: L[t-w:t]
    srt = np.sort(win, axis=1)
    o = out['HS']
    for a, k in lev.items():
        o[k] = np.quantile(win, 1 - a, axis=1)
    for a, k in ((0.025, 'ES2.5'), (0.01, 'ES1')):
        v = np.quantile(win, 1 - a, axis=1)
        o[k] = np.nanmean(np.where(win >= v[:, None], win, np.nan), axis=1)
    o['pit'] = (np.sum(win < Lcur[:, None], axis=1) + 0.5 * np.sum(win == Lcur[:, None], axis=1)) / w
    o['scale'] = win.std(axis=1, ddof=1)
    o['mu'] = np.zeros(T)
    tails['HS'] = srt[:, -K_TAIL:]
    mu_w, sd_w = -win.mean(axis=1), win.std(axis=1, ddof=1)     # mean of returns, standard deviation
    o = out['Normal']
    for a, k in lev.items():
        o[k] = -mu_w + sd_w * stats.norm.ppf(1 - a)
    for a, k in ((0.025, 'ES2.5'), (0.01, 'ES1')):
        o[k] = -mu_w + sd_w * stats.norm.pdf(stats.norm.ppf(1 - a)) / a
    o['pit'] = stats.norm.cdf((Lcur + mu_w) / sd_w)
    o['scale'], o['mu'] = sd_w, mu_w

    # --- models re-estimated every `refit` days
    prev = None
    for b0 in range(0, T, refit):
        t0 = start + b0
        b1 = min(T, b0 + refit)
        x = r.values[t0 - w:t0]
        # unconditional Student-t: degrees of freedom, location and scale by MLE
        nu_t, loc_t, sc_t = stats.t.fit(x)
        nu_t = max(nu_t, 2.2)
        sd_t = sc_t * np.sqrt(nu_t / (nu_t - 2))                # implied standard deviation
        # GARCH(1,1)-t
        prev = garch_fit(x, prev if b0 > 0 else None)
        mu, om, al, be, nu = prev
        nu = max(nu, 2.2)
        xx = r.values[t0 - w:t0 + (b1 - b0)]
        s2 = garch_filter(xx, mu, om, al, be, np.var(x[:50]))
        z = (x - mu) / np.sqrt(s2[:w])
        y = np.sort(-z)                                         # standardised losses
        g = gpd_tail(y, (0.01, 0.025, 0.05))
        for t in range(b0, b1):
            j = w + (t - b0)
            sig = np.sqrt(s2[j])
            Lt = Lcur[t]
            # unconditional Student-t
            o = out['Student-t']
            for a, k in lev.items():
                o[k][t] = -loc_t + sd_t * t_std_q(nu_t, a)
            o['ES2.5'][t] = -loc_t + sd_t * t_std_es(nu_t, 0.025)
            o['ES1'][t] = -loc_t + sd_t * t_std_es(nu_t, 0.01)
            o['pit'][t] = stats.t.cdf((Lt + loc_t) / sc_t, nu_t)
            o['scale'][t], o['mu'][t], o['nu'][t] = sd_t, loc_t, nu_t
            # GARCH-t
            o = out['GARCH-t']
            for a, k in lev.items():
                o[k][t] = -mu + sig * t_std_q(nu, a)
            o['ES2.5'][t] = -mu + sig * t_std_es(nu, 0.025)
            o['ES1'][t] = -mu + sig * t_std_es(nu, 0.01)
            o['pit'][t] = stats.t.cdf((Lt + mu) / sig / np.sqrt((nu - 2) / nu), nu)
            o['scale'][t], o['mu'][t], o['nu'][t] = sig, mu, nu
            # FHS: empirical quantiles of the standardised losses
            o = out['FHS']
            for a, k in lev.items():
                o[k][t] = -mu + sig * np.quantile(y, 1 - a)
            for a, k in ((0.025, 'ES2.5'), (0.01, 'ES1')):
                q = np.quantile(y, 1 - a)
                o[k][t] = -mu + sig * y[y >= q].mean()
            o['pit'][t] = np.mean(y < (Lt + mu) / sig)
            o['scale'][t], o['mu'][t] = sig, mu
            tails['FHS'][t] = -mu + sig * y[-K_TAIL:]
            # GARCH-EVT
            o = out['GARCH-EVT']
            for a, k in lev.items():
                o[k][t] = -mu + sig * g['var'][a]
            o['ES2.5'][t] = -mu + sig * g['es'][0.025]
            o['ES1'][t] = -mu + sig * g['es'][0.01]
            o['pit'][t] = gpd_cdf((Lt + mu) / sig, g, y)
            o['scale'][t], o['mu'][t] = sig, mu
            o['u'][t], o['xi'][t], o['beta'][t], o['pu'][t] = g['u'], g['xi'], g['beta'], g['pu']
    res = {}
    for m in MODELS:
        df = pd.DataFrame(out[m], index=days)
        df['L'] = Lcur
        res[m] = df
    return res, tails


def get_forecasts(name):
    """Forecasts for one asset, with a temporary cache (the computation takes a few minutes)."""
    path = os.path.join(tempfile.gettempdir(), f'mfm_ch8_fc_{name}_{W}_{REFIT}.pkl')
    if os.path.exists(path):
        with open(path, 'rb') as f:
            return pickle.load(f)
    fc = rolling_forecasts(load_returns(name))
    with open(path, 'wb') as f:
        pickle.dump(fc, f)
    return fc


# =============================================================================
# VaR TESTS
# =============================================================================
def kupiec(hits, p):
    """Kupiec POF test: LR_uc ~ chi2(1)."""
    hits = np.asarray(hits, int)
    T, x = len(hits), hits.sum()
    ph = x / T
    ll0 = (T - x) * np.log(1 - p) + x * np.log(p)
    ll1 = (T - x) * np.log(1 - ph) + x * np.log(ph) if 0 < x < T else 0.0
    lr = -2 * (ll0 - ll1)
    return dict(T=T, x=int(x), rate=ph, LR=lr, p=stats.chi2.sf(lr, 1))


def transitions(hits):
    """Counts of the transitions n00, n01, n10, n11 of the hit sequence."""
    h = np.asarray(hits, int)
    a, b = h[:-1], h[1:]
    return dict(n00=int(np.sum((a == 0) & (b == 0))), n01=int(np.sum((a == 0) & (b == 1))),
                n10=int(np.sum((a == 1) & (b == 0))), n11=int(np.sum((a == 1) & (b == 1))))


def christoffersen_from_counts(n00, n01, n10, n11, p):
    """LR_ind (independence, first-order Markov chain) and LR_cc = LR_uc + LR_ind."""
    def xlogy(n, q):
        return n * np.log(q) if n > 0 else 0.0
    pi01 = n01 / (n00 + n01)
    pi11 = n11 / (n10 + n11) if (n10 + n11) > 0 else 0.0
    pi = (n01 + n11) / (n00 + n01 + n10 + n11)
    l_ind = xlogy(n00, 1 - pi01) + xlogy(n01, pi01) + xlogy(n10, 1 - pi11) + xlogy(n11, pi11)
    l_0 = xlogy(n00 + n10, 1 - pi) + xlogy(n01 + n11, pi)
    lr_ind = -2 * (l_0 - l_ind)
    T, x = n00 + n01 + n10 + n11, n01 + n11
    lr_uc = -2 * (xlogy(T - x, 1 - p) + xlogy(x, p) - xlogy(T - x, 1 - pi) - xlogy(x, pi))
    lr_cc = lr_uc + lr_ind
    return dict(pi01=pi01, pi11=pi11, pi=pi, LR_ind=lr_ind, p_ind=stats.chi2.sf(lr_ind, 1),
                LR_uc=lr_uc, LR_cc=lr_cc, p_cc=stats.chi2.sf(lr_cc, 2))


def christoffersen(hits, p):
    return christoffersen_from_counts(**transitions(hits), p=p)


def durations(hits):
    """Durations (days) between breaches, with censoring flags for the first and last duration."""
    h = np.asarray(hits, int)
    pos = np.flatnonzero(h)
    if len(pos) == 0:
        return np.array([len(h)]), 1, 1
    D = np.diff(np.r_[-1, pos, len(h) - 1 if h[-1] == 0 else pos[-1]])
    D = D[D > 0]
    c1 = 1 if h[0] == 0 else 0
    cn = 1 if h[-1] == 0 else 0
    return D.astype(float), c1, cn


def duration_test(hits):
    """Christoffersen-Pelletier (2004) duration test: continuous Weibull approximation, H0: b = 1 (memoryless)."""
    D, c1, cn = durations(hits)
    if len(D) < 3:
        return dict(b=np.nan, LR=np.nan, p=np.nan, n=len(D))

    def nll(par, fix_b=False):
        a = np.exp(par[0])
        b = 1.0 if fix_b else np.exp(par[1])
        lf = np.log(b) + b * np.log(a) + (b - 1) * np.log(D) - (a * D) ** b
        ls = -(a * D) ** b
        ll = lf.copy()
        if c1:
            ll[0] = ls[0]
        if cn:
            ll[-1] = ls[-1]
        return -ll.sum()
    a0 = np.log(1 / D.mean())
    r1 = optimize.minimize(nll, [a0, 0.0], method='Nelder-Mead', options=dict(xatol=1e-8, fatol=1e-10, maxiter=4000))
    r0 = optimize.minimize(lambda p: nll(p, True), [a0], method='Nelder-Mead')
    lr = max(2 * (r0.fun - r1.fun), 0.0)
    return dict(b=float(np.exp(r1.x[1])), LR=lr, p=stats.chi2.sf(lr, 1), n=len(D))


def traffic_light(x, T=250, p=0.01):
    """Basel zone (green/yellow/red) for x exceptions in T days; cumulative Binomial probability."""
    cum = stats.binom.cdf(x, T, p)
    zone = 'green' if cum < 0.95 else ('yellow' if cum < 0.9999 else 'red')
    return zone, cum


# =============================================================================
# ES TESTS
# =============================================================================
def mcneil_frey(L, var, es, scale, B=5000, seed=0):
    """Exceedance residuals (L - ES)/sigma on days with L > VaR; H0: mean 0, H1: mean > 0 (i.i.d. bootstrap)."""
    hit = L > var
    e = ((L - es) / scale)[hit]
    n = len(e)
    if n < 3:
        return dict(n=n, mean=np.nan, t=np.nan, p=np.nan)
    tstat = e.mean() / (e.std(ddof=1) / np.sqrt(n))
    rng = np.random.default_rng(seed)
    ec = e - e.mean()
    bs = rng.choice(ec, (B, n), replace=True)
    tb = bs.mean(1) / (bs.std(1, ddof=1) / np.sqrt(n))
    return dict(n=n, mean=e.mean(), t=tstat, p=float(np.mean(tb >= tstat)))


def acerbi_szekely(L, var, es, a=A_ES):
    """Acerbi-Szekely (2014) Z1 and Z2 statistics, with losses positive."""
    hit = L > var
    T, n = len(L), hit.sum()
    z1 = 1 - np.mean(L[hit] / es[hit]) if n > 0 else np.nan
    z2 = 1 - np.sum(L[hit] / es[hit]) / (T * a)
    return z1, z2


def tail_sampler(df, model, tails, U):
    """Inverse loss distribution function for levels U > 1 - A_ES (matrix T x M)."""
    mu, sc = df['mu'].values[:, None], df['scale'].values[:, None]
    if model in ('HS', 'FHS'):
        k = np.clip(np.ceil((U - (1 - A_ES)) / A_ES * K_TAIL).astype(int) - 1, 0, K_TAIL - 1)
        return np.take_along_axis(tails[model], k, axis=1)
    if model == 'Normal':
        return -mu + sc * stats.norm.ppf(U)
    if model in ('Student-t', 'GARCH-t'):
        nu = df['nu'].values[:, None]
        return -mu + sc * stats.t.ppf(U, nu) * np.sqrt((nu - 2) / nu)
    if model == 'GARCH-EVT':
        u, xi, be, pu = (df[c].values[:, None] for c in ('u', 'xi', 'beta', 'pu'))
        return -mu + sc * (u + be / xi * (((1 - U) / pu) ** (-xi) - 1))
    raise ValueError(model)


def z_pvalues(df, model, tails, M=2000, seed=1, a=A_ES):
    """Simulated p-values under H0 (the model's forecast distribution) for Z1 and Z2."""
    L, var, es = df['L'].values, df['VaR2.5'].values, df['ES2.5'].values
    z1, z2 = acerbi_szekely(L, var, es, a)
    rng = np.random.default_rng(seed)
    T = len(L)
    z1s, z2s = np.empty(M), np.empty(M)
    step = 250
    for m0 in range(0, M, step):
        k = min(step, M - m0)
        U = rng.uniform(size=(T, k))
        hit = U > 1 - a
        Us = np.where(hit, U, 1 - a / 2)
        Ls = tail_sampler(df, model, tails, Us)
        ratio = np.where(hit, Ls / es[:, None], 0.0)
        z2s[m0:m0 + k] = 1 - ratio.sum(0) / (T * a)
        nh = hit.sum(0)
        z1s[m0:m0 + k] = 1 - ratio.sum(0) / np.maximum(nh, 1)
    return dict(Z1=z1, Z2=z2, p1=float(np.mean(z1s <= z1)), p2=float(np.mean(z2s <= z2)),
                crit2=float(np.quantile(z2s, 0.05)))


def du_escanciano(pit, a=A_ES, lags=5):
    """Du-Escanciano (2017): cumulative violation H_t = (1/a)(u_t - (1-a)) 1{u_t > 1-a};
    unconditional test U_ES ~ N(0,1) and conditional test C_ES (Box-Pierce, chi2(lags))."""
    u = np.clip(np.asarray(pit), 0, 1)
    H = (u - (1 - a)) / a * (u > 1 - a)
    T = len(H)
    U = np.sqrt(T) * (H.mean() - a / 2) / np.sqrt(a * (1 / 3 - a / 4))
    hc = H - a / 2
    g0 = np.mean(hc ** 2)
    rho = np.array([np.mean(hc[j:] * hc[:-j]) / g0 for j in range(1, lags + 1)])
    C = T * np.sum(rho ** 2)
    return dict(meanH=H.mean(), U=U, pU=2 * stats.norm.sf(abs(U)), C=C, pC=stats.chi2.sf(C, lags))


# =============================================================================
# FZ0 LOSS, DIEBOLD-MARIANO, MCS
# =============================================================================
def fz0(L, var, es, a=A_ES):
    """FZ0 loss (Patton, Ziegel & Chen, 2019), written for positive losses:
    FZ0 = (1/(a ES)) 1{L > VaR} (L - VaR) + VaR/ES + log(ES) - 1  (ES > 0)."""
    return (L > var) * (L - var) / (a * es) + var / es + np.log(es) - 1


def pinball(L, var, a):
    """Quantile (tick) loss: (1{L > VaR} - a)(L - VaR) >= 0, minimised in expectation by the 1-a quantile."""
    return ((L > var).astype(float) - a) * (L - var)


def nw_var(d, lags=None):
    """Long-run variance (Newey-West, Bartlett kernel)."""
    d = np.asarray(d) - np.mean(d)
    T = len(d)
    if lags is None:
        lags = int(np.floor(4 * (T / 100) ** (2 / 9)))
    v = np.mean(d ** 2)
    for j in range(1, lags + 1):
        v += 2 * (1 - j / (lags + 1)) * np.dot(d[j:], d[:-j]) / T     # autocovariance with divisor T (PSD)
    return v


def diebold_mariano(la, lb, lags=None):
    """DM = mean(d) / sqrt(LRV/T), d = la - lb; negative => A has the lower average loss."""
    d = np.asarray(la) - np.asarray(lb)
    dm = d.mean() / np.sqrt(nw_var(d, lags) / len(d))
    return dict(dbar=d.mean(), DM=dm, p=2 * stats.norm.sf(abs(dm)))


def mcs(losses, alpha=0.10, B=1000, block=10, seed=2):
    """Model confidence set (Hansen, Lunde & Nason, 2011), T_max statistic,
    circular block bootstrap. losses: DataFrame T x m. Result: MCS p-values."""
    rng = np.random.default_rng(seed)
    X = losses.values
    T, m = X.shape
    nb = int(np.ceil(T / block))
    idx = (rng.integers(0, T, (B, nb))[:, :, None] + np.arange(block)) % T
    idx = idx.reshape(B, -1)[:, :T]
    Lbar = X.mean(0)
    Lb = np.stack([X[i].mean(0) for i in idx])           # B x m
    alive = list(range(m))
    pvals, p_run = {}, 0.0
    while len(alive) > 1:
        a = np.array(alive)
        d = Lbar[a] - Lbar[a].mean()
        db = Lb[:, a] - Lb[:, a].mean(1, keepdims=True)
        se = np.sqrt(np.mean((db - d) ** 2, axis=0))
        t = d / se
        tb = ((db - d) / se).max(1)
        p = float(np.mean(tb >= t.max()))
        p_run = max(p_run, p)
        worst = a[np.argmax(t)]
        pvals[losses.columns[worst]] = p_run
        alive.remove(worst)
    pvals[losses.columns[alive[0]]] = 1.0
    return pd.Series(pvals)[losses.columns]


# =============================================================================
# CONFORMAL PREDICTION FOR VaR
# =============================================================================
def conformal_var(r, a=0.01, n_cal=250, n_sd=250, gamma=0.005, eval_from=EVAL_FROM):
    """Rolling split conformal VaR and ACI (Gibbs & Candes, 2021).

    Nonconformity score: s_t = L_t / sd_t, sd_t = standard deviation of the previous n_sd returns.
    VaR_t = sd_t * the order statistic k = ceil((n+1)(1-a)) of the last n_cal scores (+inf if k > n).
    ACI: a_{t+1} = a_t + gamma (a - err_t); if the required rank exceeds n, VaR = +inf.
    Rolling-standardised scores are only approximately exchangeable, even for i.i.d. returns."""
    r = r.dropna()
    L = -r.values
    sd = pd.Series(r.values).rolling(n_sd).std().shift(1).values   # only information up to t-1
    s = L / sd
    start = max(n_sd + n_cal + 1, int(np.searchsorted(r.index, pd.Timestamp(eval_from))))
    T = len(r) - start
    var_s, var_a, a_path = np.empty(T), np.empty(T), np.empty(T)
    at = a
    for i in range(T):
        t = start + i
        cal = np.sort(s[t - n_cal:t])
        k = int(np.ceil((n_cal + 1) * (1 - a)))
        var_s[i] = np.inf if k > n_cal else sd[t] * cal[k - 1]
        ka = int(np.ceil((n_cal + 1) * (1 - at)))
        var_a[i] = np.inf if ka > n_cal else sd[t] * cal[max(ka, 1) - 1]
        a_path[i] = at
        err = float(L[t] > var_a[i])
        at = at + gamma * (a - err)
    return pd.DataFrame({'L': L[start:], 'VaR_split': var_s, 'VaR_aci': var_a, 'alpha_t': a_path,
                         'sd': sd[start:]}, index=r.index[start:])


def regimes(r, idx, n=20, q=0.90, w=W, min_obs=250):
    """Crisis days: volatility over the n days before day t (excluding day t) above the q quantile of the
    same volatilities over the last w days (the models' estimation window, at least min_obs values);
    the indicator and its threshold use returns up to t-1 only, so they are known ex ante."""
    rv = r.rolling(n).std().shift(1)
    cut = rv.rolling(w, min_periods=min_obs).quantile(q)
    return (rv > cut).reindex(idx).fillna(False).astype(bool)

## Style and helpers

In [ ]:
# MFM standard style (as in SFM): transparent + English labels + legend at the bottom
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 9
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['lines.linewidth'] = 1.1
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['legend.fontsize'] = 8

# Brand colours
MainBlue = '#1A3A6E'
IDAred   = '#CD0000'
Forest   = '#2E7D32'
Amber    = '#B5853F'
Orange   = '#E67E22'
Purple   = '#8E44AD'
Crimson  = '#DC3545'
Gray     = '#7F7F7F'
LightGray = '#DADADA'
Teal = '#17A2B8'
MCOL = {'HS': Teal, 'Normal': Amber, 'Student-t': Purple, 'GARCH-t': MainBlue, 'FHS': Forest, 'GARCH-EVT': IDAred}
PV_CMAP = LinearSegmentedColormap.from_list('pv', [IDAred, '#F4B6B6', '#FFFFFF', '#BFD8BF', Forest])


def save_fig(name):
    """Salveaza figura ca PDF si PNG transparent, in folderul curent."""
    plt.savefig(f'{name}.pdf', bbox_inches='tight', transparent=True)
    plt.savefig(f'{name}.png', bbox_inches='tight', transparent=True, dpi=180)
    plt.show()
    plt.close()
    print(f"   saved {name}")


_FC = {}


def legend_outside_bottom(ax, ncol=2, y=-0.22):
    """Place the legend outside the plot, bottom centre."""
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def pv_heatmap(ax, P, fmt='{:.2f}', title=None):
    """Heat map of p-values: red = rejection, green = no rejection (log scale)."""
    Z = np.log10(np.clip(P.values.astype(float), 1e-4, 1))
    ax.imshow(Z, cmap=PV_CMAP, vmin=-4, vmax=0, aspect='auto')
    for i in range(P.shape[0]):
        for j in range(P.shape[1]):
            v = P.values[i, j]
            txt = '<0.001' if v < 0.001 else (f'{v:.3f}' if v < 0.01 else fmt.format(v))
            ax.text(j, i, txt, ha='center', va='center', fontsize=7)
    ax.set_xticks(range(P.shape[1]), P.columns, fontsize=7.5)
    ax.set_yticks(range(P.shape[0]), P.index, fontsize=7.5)
    for s in ax.spines.values():
        s.set_visible(False)
    if title:
        ax.set_title(title, fontsize=9)


def fc(name):
    if name not in _FC:
        _FC[name] = get_forecasts(name)
    return _FC[name]


def hits(name, model, col='VaR1'):
    df = fc(name)[0][model]
    return (df['L'] > df[col]).astype(int)

A6_L = np.array([np.float64(0.8), np.float64(-1.1), np.float64(3.4), np.float64(2.3), np.float64(-0.5)])
A6_FC = {'A': (2.0, 2.6), 'B': (2.6, 3.4), 'C': (2.0, 4.0)}
A7_L = np.array([np.float64(3.9), np.float64(2.8), np.float64(4.6), np.float64(3.1), np.float64(5.2), np.float64(2.9), np.float64(3.6), np.float64(7.4)])
A7_ES = np.array([np.float64(3.4), np.float64(3.3), np.float64(3.6), np.float64(3.2), np.float64(3.5), np.float64(3.3), np.float64(3.4), np.float64(3.8)])
A8_S = np.array([np.float64(1.12), np.float64(0.35), np.float64(2.41), np.float64(0.88), np.float64(1.95), np.float64(0.52), np.float64(3.1), np.float64(1.47), np.float64(0.73), np.float64(2.02), np.float64(0.19), np.float64(1.66), np.float64(2.75), np.float64(0.94), np.float64(1.28), np.float64(0.61), np.float64(1.83), np.float64(2.28), np.float64(1.05)])
A8_SD = 1.4
A3_SEQ = [0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0]

## Analysis

In [ ]:
def a1_kupiec(T=250, x=7, p=0.01):
    """Kupiec step by step."""
    ph = x / T
    l0 = (T - x) * np.log(1 - p) + x * np.log(p)
    l1 = (T - x) * np.log(1 - ph) + x * np.log(ph)
    lr = -2 * (l0 - l1)
    return dict(T=T, x=x, p=p, ph=ph, l0=l0, l1=l1, LR=lr, pval=stats.chi2.sf(lr, 1),
                exact=float(stats.binom.sf(x - 1, T, p)), expected=T * p)


def a3_counts(name, model, a='2007', b='2010', p=0.01):
    h = hits(name, model).loc[a:b]
    c = transitions(h.values)
    return dict(**c, **christoffersen_from_counts(**c, p=p), T=len(h))


def a5_traffic(T=250, p=0.01):
    cum = [float(stats.binom.cdf(k, T, p)) for k in range(0, 13)]
    return dict(cum=cum, green_max=int(np.max(np.flatnonzero(np.array(cum) < 0.95))),
                red_min=int(np.min(np.flatnonzero(np.array(cum) >= 0.9999))), p97=float(stats.binom.cdf(5, T, 0.03)))


def a6_fz0(a=0.025):
    out = {}
    for k, (v, e) in A6_FC.items():
        terms = fz0(A6_L, v, e, a)
        out[k] = dict(VaR=v, ES=e, terms=[float(t) for t in terms], mean=float(terms.mean()),
                      const=float(v / e + np.log(e) - 1), hit=[int(l > v) for l in A6_L])
    return out


def a7_z2(T=250, a=0.025):
    r = A7_L / A7_ES
    z2 = 1 - r.sum() / (T * a)
    z1 = 1 - r.mean()
    return dict(ratios=[float(x) for x in r], sum=float(r.sum()), Z2=float(z2), Z1=float(z1), n=len(r), Ta=T * a)


def a8_conformal(a=0.10, gamma=0.02):
    n = len(A8_S)
    k = int(np.ceil((n + 1) * (1 - a)))
    s = np.sort(A8_S)
    q = s[k - 1]
    a_next_hit = a + gamma * (a - 1)
    a_next_nohit = a + gamma * a
    k_hit = int(np.ceil((n + 1) * (1 - a_next_hit)))
    return dict(n=n, k=k, q=float(q), VaR=float(A8_SD * q), sorted=[float(x) for x in s],
                a_hit=a_next_hit, a_nohit=a_next_nohit, k_hit=k_hit, q_hit=float(s[min(k_hit, n) - 1]) if k_hit <= n else None,
                k_nohit=int(np.ceil((n + 1) * (1 - a_next_nohit))))


def part_a():
    return dict(a1=a1_kupiec(), a2=a1_kupiec(250, 11, 0.025), a3=a3_counts('bet', 'HS'), a4=a3_counts('bet', 'GARCH-t'),
                a5=a5_traffic(), a6=a6_fz0(), a7=a7_z2(), a8=a8_conformal())


def pof_power(T, pis, a=0.01, level=0.05):
    """Exact power of the Kupiec test (LR, 5% level) under Binomial(T, pi) and the local
    noncentral chi2 approximation with parameter T (pi - a)^2 / (a(1-a))."""
    x = np.arange(T + 1)
    lr = np.array([kupiec(np.r_[np.ones(k), np.zeros(T - k)], a)['LR'] for k in x])
    rej = lr > stats.chi2.ppf(1 - level, 1)
    exact = np.array([stats.binom.pmf(x, T, p)[rej].sum() for p in pis])
    lam = T * (pis - a) ** 2 / (a * (1 - a))
    asym = stats.ncx2.sf(stats.chi2.ppf(1 - level, 1), 1, np.maximum(lam, 1e-12))
    return exact, asym, rej


def a1_pof_power(T=250, pi=0.02, a=0.01):
    """Local (noncentral chi2) and exact (Binomial) power of the 5% Kupiec test; exact size."""
    lam = T * (pi - a) ** 2 / (a * (1 - a))
    crit = stats.chi2.ppf(0.95, 1)
    ex, asy, rej = pof_power(T, np.array([pi, a]), a)
    return dict(T=T, pi=pi, a=a, lam=float(lam), asym=float(stats.ncx2.sf(crit, 1, lam)), exact=float(ex[0]),
                size=float(ex[1]), accept=[int(k) for k in np.flatnonzero(~rej)[[0, -1]]])


def a2_days_for_power(pi=0.05, a=0.025, power=0.80):
    """Number of days for a given power: from lambda* (asymptotic) and by exact search."""
    from scipy.optimize import brentq
    crit = stats.chi2.ppf(0.95, 1)
    lam = brentq(lambda l: stats.ncx2.sf(crit, 1, l) - power, 0.1, 50)
    T_asy = lam * a * (1 - a) / (pi - a) ** 2
    T_ex = next(T for T in range(20, 2000) if pof_power(T, np.array([pi]), a)[0][0] >= power)
    return dict(lam=float(lam), T_asy=float(T_asy), T_exact=int(T_ex))


def a4_dq_constant(T=250, x=7, a=0.01):
    """DQ with X_t = 1: score form (Wald with the null variance) of the Kupiec test, versus LR."""
    ph = x / T
    dq = T * (ph - a) ** 2 / (a * (1 - a))
    k = a1_kupiec(T, x, a)
    return dict(DQ=float(dq), p=float(stats.chi2.sf(dq, 1)), LR=float(k['LR']), p_LR=float(k['pval']))


def a6_consistency(nu=4, a=0.025, n=1_000_000, seed=5):
    """Numerical check: minimisers of the average pinball and FZ0 losses on unit-variance t(nu) losses."""
    from scipy.optimize import minimize, minimize_scalar
    rng = np.random.default_rng(seed)
    L = rng.standard_t(nu, n) * np.sqrt((nu - 2) / nu)
    v_pin = minimize_scalar(lambda v: pinball(L, v, a).mean(), bounds=(0.5, 5), method='bounded').x
    r = minimize(lambda p: fz0(L, p[0], p[1], a).mean() if p[1] >= p[0] > 0 else 1e9, [2.0, 2.5],
                 method='Nelder-Mead', options=dict(xatol=1e-5, fatol=1e-9))
    return dict(v_pin=float(v_pin), v_fz=float(r.x[0]), e_fz=float(r.x[1]),
                VaR=float(t_std_q(nu, a)), ES=float(t_std_es(nu, a)))


def es_discrete(vals, probs, a):
    """ES at tail probability a for a discrete loss distribution."""
    o = np.argsort(vals)[::-1]
    v, p = np.asarray(vals, float)[o], np.asarray(probs, float)[o]
    take = np.minimum(p, np.maximum(a - np.r_[0, np.cumsum(p)[:-1]], 0))
    return float((v * take).sum() / a)


def a9_es_not_elicitable(a=0.025):
    """Two distributions with the same ES whose mixture has a different ES: level sets are not convex."""
    F0 = ([1.0], [1.0])
    F1 = ([2.0, 0.0], [a / 2, 1 - a / 2])
    mix = ([2.0, 1.0, 0.0], [a / 4, 0.5, 0.5 - a / 4])
    return dict(es0=es_discrete(*F0, a), es1=es_discrete(*F1, a), es_mix=es_discrete(*mix, a))


def a10_de_moments(a=0.025, n=4_000_000, seed=6, T=250):
    """Moments of the cumulative violation H = (u - (1 - a))/a 1{u > 1 - a} under H0 (u uniform)."""
    u = np.random.default_rng(seed).uniform(size=n)
    H = (u - (1 - a)) / a * (u > 1 - a)
    return dict(mean=float(H.mean()), var=float(H.var()), mean_th=a / 2, var_th=a * (1 / 3 - a / 4),
                se_T=float(np.sqrt(a * (1 / 3 - a / 4) / T)))


def part_a_extras():
    return dict(a1p=a1_pof_power(), a2p=a1_pof_power(250, 0.05, 0.025), a2T=a2_days_for_power(),
                a4dq=a4_dq_constant(), a6c=a6_consistency(), a9=a9_es_not_elicitable(), a10=a10_de_moments())


def _bars_legend(fig, axs, ncol=4, y=0.0):
    hs, ls = [], []
    for ax in np.atleast_1d(axs):
        h, l = ax.get_legend_handles_labels()
        for hh, ll in zip(h, l):
            if ll not in ls:
                hs.append(hh)
                ls.append(ll)
    fig.legend(hs, ls, loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def ch_a1():
    """A1: Binomial probabilities under the null (1%) and the alternative (2%), LR rejection region; power curves."""
    T, a = 250, 0.01
    xs = np.arange(0, 13)
    ex, asy, rej = pof_power(T, np.array([0.02, a]), a)
    pis = np.linspace(0.0, 0.04, 161)
    exc, asyc, _ = pof_power(T, pis, a)
    fig, axs = plt.subplots(1, 2, figsize=(7.4, 2.8))
    ax = axs[0]
    for x in xs[rej[:13]]:
        ax.axvspan(x - 0.5, x + 0.5, color=IDAred, alpha=0.08, lw=0)
    ax.bar(xs - 0.2, stats.binom.pmf(xs, T, a), 0.4, color=MainBlue, label='P(x), correct model, π = 1%')
    ax.bar(xs + 0.2, stats.binom.pmf(xs, T, 0.02), 0.4, color=IDAred, label='P(x), true π = 2%')
    ax.set_xticks(xs)
    ax.set_xlabel('Breaches x in T = 250 days (shaded: LR rejects at 5%)')
    ax.set_ylabel('Probability')
    ax = axs[1]
    ax.plot(100 * pis, exc, color=MainBlue, lw=1.4, label='Exact rejection probability')
    ax.plot(100 * pis, asyc, color=Orange, lw=1.2, ls='--', label='Local asymptotic power')
    ax.axhline(0.05, color=Gray, lw=0.7, ls=':')
    ax.plot([1, 2, 2], [ex[1], ex[0], asy[0]], 'o', color='black', ms=3.5)
    ax.annotate(f'size {100 * ex[1]:.1f}%', (1, ex[1]), xytext=(1.15, 0.22), fontsize=7.5, color='black',
                arrowprops=dict(arrowstyle='-', lw=0.6, color='black'))
    ax.annotate(f'exact {100 * ex[0]:.0f}%', (2, ex[0]), xytext=(2.35, 0.12), fontsize=7.5, color='black',
                arrowprops=dict(arrowstyle='-', lw=0.6, color='black'))
    ax.annotate(f'local {100 * asy[0]:.0f}%', (2, asy[0]), xytext=(2.35, 0.45), fontsize=7.5, color='black',
                arrowprops=dict(arrowstyle='-', lw=0.6, color='black'))
    ax.set_xlabel('True breach probability π (%)')
    ax.set_ylabel('Rejection probability, T = 250')
    ax.set_ylim(0, 1.02)
    plt.tight_layout()
    _bars_legend(fig, axs, ncol=4)
    save_fig('ch8_sem_a1_power')
    return dict(p0=float(stats.binom.pmf(0, T, a)), p7=float(stats.binom.sf(6, T, a)),
                pmf7_0=float(stats.binom.pmf(7, T, a)), pmf7_1=float(stats.binom.pmf(7, T, 0.02)))


def ch_a2():
    """A2: exact power against the sample size T (VaR 2.5%, true rate 5%); first 80% crossing."""
    a, pi = 0.025, 0.05
    Ts = np.arange(100, 701)
    ex = np.array([pof_power(T, np.array([pi]), a)[0][0] for T in Ts])
    lam = Ts * (pi - a) ** 2 / (a * (1 - a))
    asy = stats.ncx2.sf(stats.chi2.ppf(0.95, 1), 1, lam)
    first = int(Ts[np.argmax(ex >= 0.8)])
    below_after = Ts[(Ts > first) & (ex < 0.8)]
    fig, ax = plt.subplots(figsize=(6.6, 2.8))
    ax.plot(Ts, ex, color=MainBlue, lw=1.0, label='Exact power (Binomial, LR rule at 5%)')
    ax.plot(Ts, asy, color=Orange, lw=1.2, ls='--', label='Local asymptotic power')
    ax.axhline(0.8, color=Gray, lw=0.7, ls=':')
    ax.axvline(first, color=IDAred, lw=0.9, ls='--', label=f'First exact crossing of 80%: T = {first}')
    T_asy = int(np.ceil(Ts[np.argmax(asy >= 0.8)]))
    ax.axvline(T_asy, color=Forest, lw=0.9, ls='--', label=f'Asymptotic 80%: T = {T_asy}')
    ax.set_xlabel('Number of days T')
    ax.set_ylabel('Power against π = 5%')
    legend_outside_bottom(ax, ncol=2, y=-0.2)
    save_fig('ch8_sem_a2_power')
    return dict(first=first, T_asy=T_asy, n_below_after=int(len(below_after)),
                last_below=int(below_after.max()) if len(below_after) else None,
                pow_first=float(ex[Ts == first][0]), pow_prev=float(ex[Ts == first - 1][0]))


def ch_a3():
    """A3: BET, HS VaR 1% in 2007-2010: losses, VaR and breaches; breach probability after a calm day and after a breach."""
    F = fc('bet')[0]['HS'].loc['2007':'2010']
    h = (F['L'] > F['VaR1'])
    c = transitions(h.astype(int).values)
    toy = transitions(A3_SEQ)
    fig, axs = plt.subplots(1, 2, figsize=(7.6, 2.8), gridspec_kw=dict(width_ratios=[2.6, 1]))
    ax = axs[0]
    ax.plot(F.index, 100 * F['L'], color=Orange, lw=0.5, label='Daily loss, BET')
    ax.plot(F.index, 100 * F['VaR1'], color=Teal, lw=1.1, label='HS VaR 1% (1000 days)')
    ax.plot(F.index[h], 100 * F['L'][h], 'o', color=IDAred, ms=3, label=f'Breach ({int(h.sum())} days)')
    ax.set_ylabel('Loss (%)')
    ax.set_ylim(-8, 14)
    ax = axs[1]
    p01, p11 = c['n01'] / (c['n00'] + c['n01']), c['n11'] / (c['n10'] + c['n11'])
    ax.bar([0], [100 * p01], 0.6, color=MainBlue, label=f'After no breach: {c["n01"]}/{c["n00"] + c["n01"]}')
    ax.bar([1], [100 * p11], 0.6, color=IDAred, label=f'After a breach: {c["n11"]}/{c["n10"] + c["n11"]}')
    ax.axhline(1.0, color='black', ls='--', lw=0.8, label='Target 1%')
    ax.set_xticks([0, 1], [r'$\hat\pi_{01}$', r'$\hat\pi_{11}$'])
    ax.set_ylabel('Breach probability (%)')
    for x, v in ((0, p01), (1, p11)):
        ax.text(x, 100 * v + 0.3, f'{100 * v:.1f}%', ha='center', fontsize=8, color='black')
    plt.tight_layout()
    _bars_legend(fig, axs, ncol=3)
    save_fig('ch8_sem_a3_bet')
    return dict(toy=toy, toy_seq=A3_SEQ, nb=int(h.sum()), start=str(F.index[0].date()), end=str(F.index[-1].date()))


def ch_a4():
    """A4: LR_uc and the intercept-only DQ (score form) against the number of breaches, T = 250, alpha = 1%."""
    T, a = 250, 0.01
    xs = np.arange(0, 16)
    lr = np.array([kupiec(np.r_[np.ones(x), np.zeros(T - x)], a)['LR'] for x in xs])
    dq = T * (xs / T - a) ** 2 / (a * (1 - a))
    c = stats.chi2.ppf(0.95, 1)
    fig, ax = plt.subplots(figsize=(6.4, 2.8))
    ax.plot(xs, lr, 'o-', color=MainBlue, ms=3.5, lw=1.1, label=r'LR$_{uc}$ (likelihood ratio)')
    ax.plot(xs, dq, 's-', color=Orange, ms=3.5, lw=1.1, label='DQ with a constant only (score form)')
    ax.axhline(c, color='black', ls='--', lw=0.8, label='5% critical value 3.84')
    ax.axvline(7, color=IDAred, lw=0.8, ls=':', label='x = 7')
    ax.set_xlabel('Breaches x in T = 250 days')
    ax.set_ylabel('Statistic')
    ax.set_ylim(0, 25)
    ax.set_xticks(xs)
    legend_outside_bottom(ax, ncol=2, y=-0.2)
    save_fig('ch8_sem_a4_dq')
    acc_lr = xs[lr <= c]
    acc_dq = xs[dq <= c]
    return dict(acc_lr=[int(acc_lr.min()), int(acc_lr.max())], acc_dq=[int(acc_dq.min()), int(acc_dq.max())],
                dq0=float(dq[0]), lr0=float(lr[0]), dq6=float(dq[6]), lr6=float(lr[6]))


def ch_a5():
    """A5: Binomial(250, p) cumulative probabilities, p = 1% and 3%, with the Basel zone cut-offs."""
    T = 250
    xs = np.arange(0, 16)
    fig, ax = plt.subplots(figsize=(6.6, 2.8))
    ax.axvspan(-0.5, 4.5, color=Forest, alpha=0.10, lw=0)
    ax.axvspan(4.5, 9.5, color='#F1C40F', alpha=0.15, lw=0)
    ax.axvspan(9.5, 15.5, color=IDAred, alpha=0.10, lw=0)
    ax.step(xs, stats.binom.cdf(xs, T, 0.01), where='mid', color=MainBlue, lw=1.4, label='P(X ≤ x), correct model, p = 1%')
    ax.step(xs, stats.binom.cdf(xs, T, 0.03), where='mid', color=IDAred, lw=1.4, label='P(X ≤ x), true p = 3%')
    ax.axhline(0.95, color='black', ls='--', lw=0.8, label='95% (yellow from here)')
    ax.axhline(0.9999, color='black', ls=':', lw=0.9, label='99.99% (red from here)')
    ax.set_xticks(xs)
    ax.set_xlabel('Exceptions x in 250 days (background: green, yellow, red zone)')
    ax.set_ylabel('Cumulative probability')
    legend_outside_bottom(ax, ncol=2, y=-0.2)
    save_fig('ch8_sem_a5_zones')
    return dict(cdf={int(x): float(stats.binom.cdf(x, T, 0.01)) for x in range(0, 12)})


def ch_a6(ax6=None, nu=4, a=0.025, n=1_000_000, seed=5):
    """A6: expected pinball loss against v and the expected FZ0 loss over (v, e), same draws as a6_consistency."""
    rng = np.random.default_rng(seed)
    L = rng.standard_t(nu, n) * np.sqrt((nu - 2) / nu)
    vq, eq = t_std_q(nu, a), t_std_es(nu, a)
    vs = np.linspace(1.0, 3.0, 81)
    Ls = np.sort(L)
    csum = np.r_[0, np.cumsum(Ls[::-1])][::-1]            # sum of the losses at or above index i
    def tail(v):
        i = np.searchsorted(Ls, v, side='right')
        k = n - i
        return (csum[i] - v * k) / n, k / n                 # E[(L-v)+], P(L > v)
    # expected pinball: E[(1{L>v} - a)(L - v)] = E[(L-v)+] - a E[L - v]
    mL = L.mean()
    pin = np.array([tail(v)[0] - a * (mL - v) for v in vs])
    ves = np.linspace(1.3, 2.6, 60)
    ees = np.linspace(1.8, 3.6, 60)
    Z = np.empty((len(ees), len(ves)))
    for j, v in enumerate(ves):
        ep, _ = tail(v)
        Z[:, j] = ep / (a * ees) + v / ees + np.log(ees) - 1
    Z[ees[:, None] < ves[None, :]] = np.nan
    ax6 = ax6 or a6_consistency()
    fig, axs = plt.subplots(1, 2, figsize=(7.4, 2.9))
    ax = axs[0]
    ax.plot(vs, pin, color=MainBlue, lw=1.4, label='Mean pinball loss')
    ax.axvline(vq, color=IDAred, ls='--', lw=0.9, label=f'True VaR 2.5% = {vq:.3f}')
    ax.set_xlabel('Reported VaR v')
    ax.set_ylabel('Mean pinball loss')
    ax = axs[1]
    cs = ax.contour(ves, ees, Z, levels=14, cmap='viridis', linewidths=0.8)
    ax.plot(vq, eq, '*', color=IDAred, ms=11, label=f'Truth ({vq:.3f}, {eq:.3f})')
    ax.plot(ax6['v_fz'], ax6['e_fz'], 'o', mfc='none', mec='black', ms=8, label='Numerical minimiser')
    ax.set_xlabel('Reported VaR v')
    ax.set_ylabel('Reported ES e (e ≥ v)')
    plt.tight_layout()
    _bars_legend(fig, axs, ncol=4)
    save_fig('ch8_sem_a6_scores')
    return dict(VaR=float(vq), ES=float(eq), v_grid_min=float(vs[np.argmin(pin)]))


def ch_a7():
    """A7: ratio L/ES of the eight breach days and the added breach; Z2 before and after."""
    r = A7_L / A7_ES
    z2 = 1 - r.sum() / 6.25
    z2b = 1 - (r.sum() + 2.0) / 6.25
    fig, axs = plt.subplots(1, 2, figsize=(7.4, 2.7), gridspec_kw=dict(width_ratios=[2, 1]))
    ax = axs[0]
    ax.bar(np.arange(1, 9), r, 0.6, color=MainBlue, label='L/ES on the 8 breach days')
    ax.bar([9], [2.0], 0.6, color=IDAred, label='Added breach: 7/3.5 = 2')
    ax.axhline(1.0, color='black', ls='--', lw=0.8, label='L = ES')
    for i, v in enumerate(np.r_[r, 2.0], 1):
        ax.text(i, v + 0.04, f'{v:.2f}', ha='center', fontsize=7, color='black')
    ax.set_xticks(range(1, 10))
    ax.set_xlabel('Breach day')
    ax.set_ylabel('L / ES')
    ax.set_ylim(0, 2.3)
    ax = axs[1]
    ax.axhspan(-0.70, 0.5, color=Forest, alpha=0.10, lw=0)
    ax.axhspan(-1.8, -0.70, color='#F1C40F', alpha=0.15, lw=0)
    ax.axhspan(-2.5, -1.8, color=IDAred, alpha=0.10, lw=0)
    ax.bar([0], [z2], 0.5, color=MainBlue, label=f'Z2, 8 breaches = {z2:.2f}')
    ax.bar([1], [z2b], 0.5, color=IDAred, label=f'Z2, 9 breaches = {z2b:.2f}')
    ax.axhline(0, color=Gray, lw=0.6)
    ax.set_xticks([0, 1], ['8 breaches', '9 breaches'])
    ax.set_ylim(-2.0, 0.4)
    ax.set_ylabel('Z2 (zones: -0.70, -1.8)')
    plt.tight_layout()
    _bars_legend(fig, axs, ncol=3)
    save_fig('ch8_sem_a7_z2')
    return dict(contrib=[float(x / 6.25) for x in r], z2=float(z2), z2b=float(z2b))


def ch_a9(a=0.025):
    """A9: tail quantile functions on (1 - a, 1] of F0, F1 and their mixture; ES = area / a."""
    fig, axs = plt.subplots(1, 3, figsize=(7.4, 2.4), sharey=True)
    u0 = 1 - a
    cases = (('F0: loss 1 for sure', [(u0, 1, 1.0)]),
             ('F1: P(L=2) = α/2, P(L=0) = 1 − α/2', [(u0, 1 - a / 2, 0.0), (1 - a / 2, 1, 2.0)]),
             ('Mixture ½F0 + ½F1', [(u0, 1 - a / 4, 1.0), (1 - a / 4, 1, 2.0)]))
    es = []
    for ax, (title, segs) in zip(axs, cases):
        for lo, hi, q in segs:
            ax.fill_between([100 * (lo - u0) / a, 100 * (hi - u0) / a], 0, q, color=MainBlue, alpha=0.3, lw=0)
            ax.plot([100 * (lo - u0) / a, 100 * (hi - u0) / a], [q, q], color=MainBlue, lw=1.6)
        e = sum((hi - lo) * q for lo, hi, q in segs) / a
        es.append(e)
        ax.axhline(e, color=IDAred, ls='--', lw=1.0, label='ES = mean of the tail quantiles')
        ax.text(3, e + 0.08, f'ES = {e:.2f}', color=IDAred, fontsize=8)
        ax.set_title(title, fontsize=8)
        ax.set_xlabel('Position in the upper α tail (%)')
        ax.set_ylim(0, 2.3)
    axs[0].set_ylabel('Tail quantile q_u')
    plt.tight_layout()
    _bars_legend(fig, axs, ncol=2)
    save_fig('ch8_sem_a9_mass')
    return dict(es=es)


def ch_a10(a=0.025, T=250, reps=20000, seed=12):
    """A10: H(u) and the sampling distribution of the mean of H over T = 250 i.i.d. uniform PITs."""
    rng = np.random.default_rng(seed)
    U = rng.uniform(size=(reps, T))
    Hb = ((U - (1 - a)) / a * (U > 1 - a)).mean(1)
    se = np.sqrt(a * (1 / 3 - a / 4) / T)
    crit_n = a / 2 + 1.645 * se
    crit_s = float(np.quantile(Hb, 0.95))
    rej = float(np.mean(Hb > crit_n))
    fig, axs = plt.subplots(1, 2, figsize=(7.4, 2.7))
    ax = axs[0]
    u = np.linspace(0.9, 1, 400)
    ax.plot(u, (u - (1 - a)) / a * (u > 1 - a), color=MainBlue, lw=1.5, label='H(u) = (u − (1 − α))/α · 1{u > 1 − α}')
    ax.axvline(1 - a, color=Gray, lw=0.7, ls=':')
    ax.set_xlabel('PIT u')
    ax.set_ylabel('H(u)')
    ax = axs[1]
    ax.hist(Hb, bins=60, color=MainBlue, alpha=0.45, edgecolor='white', label=f'Mean of H over T = 250 ({reps} samples)')
    ax.axvline(crit_n, color=IDAred, lw=1.2, label=f'Normal 95% cut-off {crit_n:.4f}')
    ax.axvline(crit_s, color=Forest, lw=1.2, ls='--', label=f'Simulated 95th percentile {crit_s:.4f}')
    ax.set_xlabel('Mean of H')
    ax.set_ylabel('Frequency')
    plt.tight_layout()
    _bars_legend(fig, axs, ncol=2)
    save_fig('ch8_sem_a10_h')
    return dict(crit_n=float(crit_n), crit_s=crit_s, rej=rej, reps=reps, share_zero=float(np.mean(Hb == 0)))

## Run

In [ ]:
print(part_a())
print(part_a_extras())
for f in (ch_a1, ch_a2, ch_a3, ch_a4, ch_a5, ch_a6, ch_a7, ch_a9, ch_a10):
    print(f())

![ch8_sem_a1_power](./ch8_sem_a1_power.png)

Output: `ch8_sem_a1_power.pdf`

![ch8_sem_a2_power](./ch8_sem_a2_power.png)

Output: `ch8_sem_a2_power.pdf`

![ch8_sem_a3_bet](./ch8_sem_a3_bet.png)

Output: `ch8_sem_a3_bet.pdf`

![ch8_sem_a4_dq](./ch8_sem_a4_dq.png)

Output: `ch8_sem_a4_dq.pdf`

![ch8_sem_a5_zones](./ch8_sem_a5_zones.png)

Output: `ch8_sem_a5_zones.pdf`

![ch8_sem_a6_scores](./ch8_sem_a6_scores.png)

Output: `ch8_sem_a6_scores.pdf`

![ch8_sem_a7_z2](./ch8_sem_a7_z2.png)

Output: `ch8_sem_a7_z2.pdf`

![ch8_sem_a9_mass](./ch8_sem_a9_mass.png)

Output: `ch8_sem_a9_mass.pdf`

![ch8_sem_a10_h](./ch8_sem_a10_h.png)

Output: `ch8_sem_a10_h.pdf`